# Credit Scoring Capstone
## Notebook 06 — Deployment Preparation

This notebook prepares the developed credit-scoring framework for deployment.

### Objectives

- Load the trained model artifacts
- Reconstruct the complete prediction pipeline
- Create a reusable prediction function
- Generate Probability of Default estimates for new applicants
- Translate PD into scenario-based Expected Loss
- Generate indicative risk-adjusted rates
- Prepare the framework for integration with a Flask API

In [2]:
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

from pathlib import Path

In [4]:
class CreditMLP(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        return self.network(x)

## 4. Load Trained Model Components

The deployed framework combines three base learners:

1. Logistic Regression
2. XGBoost
3. PyTorch MLP

Their probability outputs are passed to a Meta-Logistic Regression model,
followed by Isotonic probability calibration.

In [7]:
baseline_lr = joblib.load(
    "../models/baseline_lr.joblib"
)

xgb_pipeline = joblib.load(
    "../models/xgb_pipeline.joblib"
)

meta_model = joblib.load(
    "../models/meta_logistic.joblib"
)

isotonic_calibrator = joblib.load(
    "../models/isotonic_calibrator.joblib"
)

mlp_preprocessor = joblib.load(
    "../models/mlp_preprocessor.joblib"
)

mlp_metadata = joblib.load(
    "../models/mlp_metadata.joblib"
)

In [9]:
mlp_model = CreditMLP(
    input_size=mlp_metadata["input_size"]
)

mlp_model.load_state_dict(
    torch.load(
        "../models/mlp_state_dict.pt",
        map_location="cpu"
    )
)

mlp_model.eval()

CreditMLP(
  (network): Sequential(
    (0): Linear(in_features=251, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [11]:
print("All model artifacts loaded.")
print("MLP input size:", mlp_metadata["input_size"])

All model artifacts loaded.
MLP input size: 251


## 5. Deployment Prediction Flow

For a new applicant, the framework follows the same modelling structure used
during model development:

\[
Applicant\ Features
\]

\[
\downarrow
\]

\[
LR,\ XGBoost,\ MLP
\]

\[
\downarrow
\]

\[
P_{LR}, P_{XGB}, P_{MLP}
\]

\[
\downarrow
\]

\[
Meta\text{-}Logistic\ Regression
\]

\[
\downarrow
\]

\[
PD_{raw}
\]

\[
\downarrow
\]

\[
Isotonic\ Calibration
\]

\[
\downarrow
\]

\[
PD_{final}
\]

The final calibrated PD can then be translated into scenario-based Expected
Loss and an indicative risk-adjusted rate.

In [14]:
def predict_pd(applicant_df):

    # Logistic Regression probability
    lr_prob = baseline_lr.predict_proba(
        applicant_df
    )[:, 1]

    # XGBoost probability
    xgb_prob = xgb_pipeline.predict_proba(
        applicant_df
    )[:, 1]

    # MLP preprocessing
    mlp_input = mlp_preprocessor.transform(
        applicant_df
    )

    mlp_input = np.asarray(
        mlp_input,
        dtype=np.float32
    )

    mlp_tensor = torch.tensor(
        mlp_input,
        dtype=torch.float32
    )

    # MLP probability
    with torch.no_grad():

        mlp_logits = mlp_model(
            mlp_tensor
        )

        mlp_prob = torch.sigmoid(
            mlp_logits
        ).cpu().numpy().ravel()

    # Construct meta-features
    meta_features = np.column_stack([
        lr_prob,
        xgb_prob,
        mlp_prob
    ])

    # Raw ensemble PD
    raw_pd = meta_model.predict_proba(
        meta_features
    )[:, 1]

    # Final calibrated PD
    final_pd = isotonic_calibrator.predict(
        raw_pd
    )

    return pd.DataFrame({
        "LR_PD": lr_prob,
        "XGB_PD": xgb_prob,
        "MLP_PD": mlp_prob,
        "Raw_Ensemble_PD": raw_pd,
        "Final_PD": final_pd
    })

In [16]:
df = pd.read_csv(
    "../data/application_clean.csv"
)

In [18]:
exclude_features = [
    "SK_ID_CURR",
    "TARGET",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

X = df.drop(
    columns=exclude_features
)

In [20]:
example_applicants = X.iloc[:5].copy()

example_results = predict_pd(
    example_applicants
)

example_results.round(4)

,LR_PD,XGB_PD,MLP_PD,Raw_Ensemble_PD,Final_PD
0,0.6160,0.5462,0.4775,0.8314,0.7368
1,0.0300,0.0378,0.0183,0.0450,0.0332
2,0.0359,0.0540,0.0939,0.0517,0.0514
3,0.0624,0.0604,0.0419,0.0556,0.0651
4,0.0766,0.1189,0.0402,0.0796,0.1029


In [22]:
def calculate_financial_outputs(
    pd_values,
    exposure,
    lgd=0.40,
    fixed_rate=0.09
):

    expected_loss = (
        pd_values
        * lgd
        * exposure
    )

    indicative_rate = (
        fixed_rate
        + pd_values * lgd
    )

    return (
        expected_loss,
        indicative_rate
    )

In [24]:
example_exposure = df.loc[
    example_applicants.index,
    "AMT_CREDIT"
].values

example_el, example_rate = (
    calculate_financial_outputs(
        example_results["Final_PD"].values,
        example_exposure
    )
)

In [26]:
example_results[
    "EAD_proxy"
] = example_exposure

example_results[
    "Expected_Loss"
] = example_el

example_results[
    "Indicative_Rate"
] = example_rate

example_results.round(4)

,LR_PD,XGB_PD,MLP_PD,Raw_Ensemble_PD,Final_PD,EAD_proxy,Expected_Loss,Indicative_Rate
0,0.6160,0.5462,0.4775,0.8314,0.7368,406597.5,119839.2632,0.3847
1,0.0300,0.0378,0.0183,0.0450,0.0332,1293502.5,17167.2221,0.1033
2,0.0359,0.0540,0.0939,0.0517,0.0514,135000.0,2775.3304,0.1106
3,0.0624,0.0604,0.0419,0.0556,0.0651,312682.5,8142.4003,0.1160
4,0.0766,0.1189,0.0402,0.0796,0.1029,513000.0,21111.1111,0.1312


In [28]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

BASE_DIR = Path.cwd().parent
MODEL_DIR = BASE_DIR / "models"
DATA_DIR = BASE_DIR / "data"

print("Project directory:", BASE_DIR)
print("Model directory:", MODEL_DIR)

Project directory: C:\Users\USER\Downloads\SEM 5\Capstone Project I\Credit_Scoring_Capstone
Model directory: C:\Users\USER\Downloads\SEM 5\Capstone Project I\Credit_Scoring_Capstone\models


In [30]:
model_files = [
    "baseline_lr.joblib",
    "xgb_pipeline.joblib",
    "meta_logistic.joblib",
    "isotonic_calibrator.joblib",
    "mlp_preprocessor.joblib",
    "mlp_metadata.joblib",
    "mlp_state_dict.pt"
]

for file in model_files:
    path = MODEL_DIR / file
    print(file, "->", path.exists())

baseline_lr.joblib -> True
xgb_pipeline.joblib -> True
meta_logistic.joblib -> True
isotonic_calibrator.joblib -> True
mlp_preprocessor.joblib -> True
mlp_metadata.joblib -> True
mlp_state_dict.pt -> True


In [32]:
class CreditMLP(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

In [34]:
baseline_lr = joblib.load(
    MODEL_DIR / "baseline_lr.joblib"
)

xgb_pipeline = joblib.load(
    MODEL_DIR / "xgb_pipeline.joblib"
)

meta_model = joblib.load(
    MODEL_DIR / "meta_logistic.joblib"
)

isotonic_calibrator = joblib.load(
    MODEL_DIR / "isotonic_calibrator.joblib"
)

mlp_preprocessor = joblib.load(
    MODEL_DIR / "mlp_preprocessor.joblib"
)

mlp_metadata = joblib.load(
    MODEL_DIR / "mlp_metadata.joblib"
)

mlp_model = CreditMLP(
    input_size=mlp_metadata["input_size"]
)

mlp_model.load_state_dict(
    torch.load(
        MODEL_DIR / "mlp_state_dict.pt",
        map_location="cpu"
    )
)

mlp_model.eval()

print("All models loaded successfully.")
print("MLP input size:", mlp_metadata["input_size"])

All models loaded successfully.
MLP input size: 251


In [36]:
def predict_pd(applicant_df):

    lr_prob = baseline_lr.predict_proba(
        applicant_df
    )[:, 1]

    xgb_prob = xgb_pipeline.predict_proba(
        applicant_df
    )[:, 1]

    mlp_input = mlp_preprocessor.transform(
        applicant_df
    )

    mlp_input = np.asarray(
        mlp_input,
        dtype=np.float32
    )

    mlp_tensor = torch.tensor(
        mlp_input,
        dtype=torch.float32
    )

    with torch.no_grad():
        mlp_logits = mlp_model(mlp_tensor)

        mlp_prob = torch.sigmoid(
            mlp_logits
        ).cpu().numpy().ravel()

    meta_features = np.column_stack([
        lr_prob,
        xgb_prob,
        mlp_prob
    ])

    raw_pd = meta_model.predict_proba(
        meta_features
    )[:, 1]

    final_pd = isotonic_calibrator.predict(
        raw_pd
    )

    return pd.DataFrame({
        "LR_PD": lr_prob,
        "XGB_PD": xgb_prob,
        "MLP_PD": mlp_prob,
        "Raw_Ensemble_PD": raw_pd,
        "Final_PD": final_pd
    })

In [38]:
df = pd.read_csv(
    DATA_DIR / "application_clean.csv"
)

exclude_features = [
    "SK_ID_CURR",
    "TARGET",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

X = df.drop(
    columns=exclude_features
)

print("Input shape:", X.shape)

Input shape: (307511, 121)


In [40]:
example_applicants = X.iloc[:5].copy()

example_results = predict_pd(
    example_applicants
)

example_results.round(4)

,LR_PD,XGB_PD,MLP_PD,Raw_Ensemble_PD,Final_PD
0,0.6160,0.5462,0.4775,0.8314,0.7368
1,0.0300,0.0378,0.0183,0.0450,0.0332
2,0.0359,0.0540,0.0939,0.0517,0.0514
3,0.0624,0.0604,0.0419,0.0556,0.0651
4,0.0766,0.1189,0.0402,0.0796,0.1029


In [42]:
def calculate_financial_outputs(
    pd_values,
    exposure,
    lgd=0.40,
    fixed_rate=0.09
):

    expected_loss = (
        pd_values
        * lgd
        * exposure
    )

    indicative_rate = (
        fixed_rate
        + pd_values * lgd
    )

    return expected_loss, indicative_rate

In [44]:
example_exposure = df.loc[
    example_applicants.index,
    "AMT_CREDIT"
].values

example_el, example_rate = (
    calculate_financial_outputs(
        example_results["Final_PD"].values,
        example_exposure
    )
)

example_results["EAD_proxy"] = example_exposure
example_results["Expected_Loss"] = example_el
example_results["Indicative_Rate"] = example_rate

example_results.round(4)

,LR_PD,XGB_PD,MLP_PD,Raw_Ensemble_PD,Final_PD,EAD_proxy,Expected_Loss,Indicative_Rate
0,0.6160,0.5462,0.4775,0.8314,0.7368,406597.5,119839.2632,0.3847
1,0.0300,0.0378,0.0183,0.0450,0.0332,1293502.5,17167.2221,0.1033
2,0.0359,0.0540,0.0939,0.0517,0.0514,135000.0,2775.3304,0.1106
3,0.0624,0.0604,0.0419,0.0556,0.0651,312682.5,8142.4003,0.1160
4,0.0766,0.1189,0.0402,0.0796,0.1029,513000.0,21111.1111,0.1312


In [46]:
import sys

print(sys.executable)

C:\Users\USER\anaconda3\envs\capstone_torch\python.exe


In [48]:
import json

sample_applicant = X.iloc[[0]].copy()

sample_json = sample_applicant.to_json(
    orient="records",
    indent=4
)

sample_path = (
    BASE_DIR
    / "deployment"
    / "sample_applicant.json"
)

with open(sample_path, "w") as f:
    f.write(sample_json)

print("Saved to:")
print(sample_path)

Saved to:
C:\Users\USER\Downloads\SEM 5\Capstone Project I\Credit_Scoring_Capstone\deployment\sample_applicant.json


In [3]:
import os

print("Notebook working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Notebook working directory:
C:\Users\USER\Downloads\SEM 5\Capstone Project I\Credit_Scoring_Capstone\notebooks

Files/folders here:
['.ipynb_checkpoints', '01_Data_EDA.ipynb', '02_Base_Models.ipynb', '03_Stacking_Calibration.ipynb', '04_Financial_Analysis.ipynb', '05_Explainability.ipynb', '06_Deployment.ipynb']
